# Extension 1 — GPT-4o Vision Modality

**Thesis Section: Chapter 6.1 — Vision modality extension**

Tests GPT-4o's vision capability on time-series line charts under three conditions:

| Variant | Chart type | Question |
|---------|-----------|----------|
| Naive   | Plain line chart | Open-ended anomaly question |
| MCQ     | Plain line chart | MCQ classification template |
| Hybrid  | Annotated chart (mean/std/spike bands) | Same MCQ as Approach 3 |

The hybrid variant tests whether delivering statistical context **visually**
(as chart annotations) is as effective as embedding it in the text prompt.

**Hypothesis**: Annotated chart + MCQ ≈ text-stats + MCQ for a vision-capable model.

**Requires**: `OPENAI_API_KEY` in `.env`

In [ ]:
import sys
sys.path.insert(0, '../..')
from dotenv import load_dotenv
load_dotenv('../../.env')

import os
import numpy as np

from src.data.timeseer_client import fetch_series_api, list_series_api
from src.data.ground_truth import GROUND_TRUTH
from src.prescreener.analyze import hybrid_analyze
from src.prompts.templates import MCQ_CATEGORIES
from src.inference.vision_infer import ask_vision_naive, ask_vision_mcq, ask_vision_hybrid
from src.parsing.extract import extract_category
from src.evaluation.metrics import compute_metrics
from src.evaluation.report import print_summary_table, save_results

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env'
os.makedirs('../../data/figures', exist_ok=True)
print('Imports OK')

In [ ]:
AREA = 'Reactor 1'
tags = list_series_api(AREA)
print(f'{len(tags)} signals in {AREA}')

In [ ]:
# ── Variant 1: Naive vision ───────────────────────────────────────
print('Running NAIVE VISION baseline...')
results_naive = []

for tag in tags:
    vals, idx = fetch_series_api(tag, AREA)
    if vals is None:
        continue

    answer = ask_vision_naive(vals, tag)

    # Keyword-based extraction for open-ended answers
    up = answer.upper()
    if any(w in up for w in ['NO ANOMALY', 'LOOKS CLEAN', 'NORMAL', 'NO SIGNIFICANT']):
        cat, lbl = 'E', 'None/Clean'
    elif 'SPIKE' in up or 'OUTLIER' in up:
        cat, lbl = 'B', 'Spikes'
    elif 'DRIFT' in up or 'GRADUAL' in up:
        cat, lbl = 'A', 'Drift'
    elif 'FLAT' in up or 'FROZEN' in up or 'CONSTANT' in up or 'STALE' in up:
        cat, lbl = 'C', 'Frozen'
    elif 'NEGATIVE' in up or 'IMPOSSIBLE' in up:
        cat, lbl = 'L', 'Intermittent'
    else:
        cat, lbl = '?', 'Unclear'

    gt = GROUND_TRUTH.get(tag, '?')
    results_naive.append({'Tag': tag, 'gt': gt, 'Category': cat, 'Label': lbl})
    print(f'  {tag:<30} → {cat}) {lbl}  (GT: {gt})')

In [ ]:
# ── Variant 2: MCQ vision ─────────────────────────────────────────
print('Running MCQ VISION baseline...')
results_mcq = []

for tag in tags:
    vals, idx = fetch_series_api(tag, AREA)
    if vals is None:
        continue

    answer = ask_vision_mcq(vals, tag, MCQ_CATEGORIES)
    cat, lbl = extract_category(answer)
    gt = GROUND_TRUTH.get(tag, '?')
    results_mcq.append({'Tag': tag, 'gt': gt, 'Category': cat, 'Label': lbl})
    print(f'  {tag:<30} → {cat}) {lbl}  (GT: {gt})')

In [ ]:
# ── Variant 3: Hybrid vision (annotated chart + Approach 3 MCQ) ───
print('Running HYBRID VISION baseline...')
results_hybrid = []

for tag in tags:
    vals, idx = fetch_series_api(tag, AREA)
    if vals is None:
        continue

    chunk, question, tname, detected, chunk_desc = hybrid_analyze(vals, idx, tag)

    # Build stats dict for chart annotation
    import pandas as pd
    s = pd.Series(vals)
    baseline_std = s.rolling(10).std().shift(5).fillna(s.std())
    stats = {
        'mean':     float(vals.mean()),
        'std':      float(vals.std()),
        'spike_hi': float(vals.mean() + 4 * vals.std()),
        'spike_lo': float(vals.mean() - 4 * vals.std()),
    }

    answer = ask_vision_hybrid(vals, tag, stats, detected, question)
    cat, lbl = extract_category(answer, detected)

    gt = GROUND_TRUTH.get(tag, '?')
    results_hybrid.append({'Tag': tag, 'gt': gt, 'Detected': ', '.join(detected),
                           'Category': cat, 'Label': lbl})
    print(f'  {tag:<30} → {cat}) {lbl}  (GT: {gt})')

In [ ]:
def acc(results):
    n = [r for r in results if r['gt'] != '?']
    if not n: return 0
    return sum(r['Category'] == r['gt'] for r in n) / len(n)

print('\n' + '='*60)
print(f'GPT-4o VISION COMPARISON — {AREA}')
print('='*60)
print(f'  Naive vision  : {acc(results_naive)*100:.1f}%')
print(f'  MCQ vision    : {acc(results_mcq)*100:.1f}%')
print(f'  Hybrid vision : {acc(results_hybrid)*100:.1f}%')
print('='*60)
print('\n  Reference (text-only from 04_gpt4o_comparison.ipynb):')
print('  Hybrid text   : see notebook 04 for comparison')

save_results(results_hybrid,
             f'../../data/gpt4o_vision_hybrid_{AREA.replace(" ","_")}.txt',
             header=f'GPT-4o Vision Hybrid | {AREA}')